# Laboratorio 7. Support Vector Machines

## Carga y visualización de los datos

En este ejemplo utilizaremos un conjunto de datos sintético con dos clases. Al tener únicamente dos variables predictoras, podemos visualizar directamente la frontera de decisión construida por el SVM.

In [ ]:
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=300, noise=0.2, random_state=42)

plt.scatter(X[:, 0], X[:, 1], c=y)
plt.xlabel("X1")
plt.ylabel("X2")
plt.show()

## SVM con kernel lineal

Primero ajustamos un SVM lineal. Este modelo busca un hiperplano que separe las clases maximizando el margen entre ellas.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
from sklearn.svm import SVC

svm_linear = SVC(kernel="linear", C=1)
svm_linear.fit(X_train, y_train)

print("Train accuracy:", svm_linear.score(X_train, y_train))
print("Test accuracy:", svm_linear.score(X_test, y_test))

## SVM con kernel RFB

Los datos no son linealmente separables. Podemos utilizar un kernel para construir una frontera de decisión no lineal sin transformar explícitamente las observaciones.

In [ ]:
svm_rbf = SVC(kernel="rbf", C=1, gamma="scale")
svm_rbf.fit(X_train, y_train)

print("Train accuracy:", svm_rbf.score(X_train, y_train))
print("Test accuracy:", svm_rbf.score(X_test, y_test))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

model = svm_linear

# Crear una malla
x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300)
x2 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)

xx, yy = np.meshgrid(x1, x2)

grid = np.c_[xx.ravel(), yy.ravel()]

# Evaluar la función de decisión
Z = model.decision_function(grid)
Z = Z.reshape(xx.shape)

# Datos
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train)

# Frontera y márgenes
plt.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"])

# Vectores de soporte
plt.scatter(
    model.support_vectors_[:, 0],
    model.support_vectors_[:, 1],
    s=150,
    facecolors="none",
    edgecolors="black",
    linewidths=1.5,
    label="Vectores de soporte"
)

plt.xlabel("X1")
plt.ylabel("X2")
plt.legend()
plt.show()

## Carga de datos

Utilizaremos un conjunto de datos real para clasificación de tumores. El objetivo es predecir si un tumor pertenece a una de dos clases a partir de características obtenidas de imágenes de células.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)
print(data.target_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Validación cruzada para $C$

Utilizaremos validación cruzada de 5 folds para comparar diferentes valores de $C$. En cada fold, la estandarización se calcula únicamente con los datos utilizados para entrenamiento.

In [ ]:
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

C_values = [0.001, 0.005, 0.01, 0.1, 1, 10]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mean_scores = []

for C in C_values:

    fold_scores = []

    for train_index, val_index in kf.split(X_train, y_train):

        # Separar fold
        X_fold_train = X_train[train_index]
        X_fold_val = X_train[val_index]

        y_fold_train = y_train[train_index]
        y_fold_val = y_train[val_index]

        # Estandarizar
        scaler = StandardScaler()

        X_fold_train = scaler.fit_transform(X_fold_train)
        X_fold_val = scaler.transform(X_fold_val)

        # Entrenar SVM
        model = SVC(kernel="linear", C=C)

        model.fit(X_fold_train, y_fold_train)

        # Evaluar
        y_pred = model.predict(X_fold_val)

        fold_scores.append(accuracy_score(y_fold_val, y_pred))

    mean_scores.append(np.mean(fold_scores))

    print(
        f"C = {C:<6}",
        f"CV accuracy = {np.mean(fold_scores):.4f}"
    )

## Comparación de los valores de $C$

In [ ]:
import matplotlib.pyplot as plt

plt.plot(C_values, mean_scores, marker="o")

plt.xlabel("C")
plt.ylabel("Accuracy promedio")
plt.title("Validación cruzada")

plt.show()

Seleccionamos el valor de $C$ con el mejor desempeño promedio en validación cruzada.

In [ ]:
best_index = np.argmax(mean_scores)
best_C = C_values[best_index]

print("Mejor C:", best_C)

## Ajuste del modelo

Una vez seleccionado el mejor valor de $C$, ajustamos el modelo utilizando todos los datos de entrenamiento y evaluamos su desempeño sobre el conjunto de prueba.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

final_model = SVC(kernel="linear", C=best_C)

final_model.fit(X_train_scaled, y_train)

## Evaluación del modelo

Evaluamos el modelo sobre el conjunto de prueba, utilizando datos que no participaron en el entrenamiento ni en la selección de $C$.

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = final_model.predict(X_test_scaled)

print("Test accuracy:", accuracy_score(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))